# 05：第二轮数据改进与三组对照

04 已完成第一轮训练。下一步保留目标写法，重点改善 **动作遗漏、直接台词归属和重复句子**。

agent 已准备 50 条训练 / 12 条验证，包含第一轮数据和 26 条新增样本；另有 10 条训练前固定的未见测试，参考答案不进入训练或推理输入。本轮数据独立导出到 `datasets/processed/teaching-v2/`。

按 **步骤 1 → 2 → 3** 执行，选择 `Python (Text LoRA - Qwen)`。你的主要工作是监督新增答案、决定采用本轮方案、审阅输出。步骤 2 的一个 Cell 自动完成训练和评估；数据编写、参数配置和报告整理已由 agent 处理。

当前交付状态：数据与 CPU 模板检查已通过，第二轮 GPU 训练尚未执行；是否改善内容，要等第二轮实际生成后评审。

In [1]:
import html
import json
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path

from IPython.display import HTML, Markdown, display

project_root = next((candidate for parent in [Path.cwd(), *Path.cwd().parents]
                     for candidate in (parent, parent / "text-lora")
                     if (candidate / "scripts/check_environment.py").is_file()), None)
if project_root is None:
    raise RuntimeError("请从当前 text-lora 项目打开 Notebook")
if str(project_root / "scripts") not in sys.path:
    sys.path.insert(0, str(project_root / "scripts"))
from check_environment import check_python
from prepare_training_data import canonical_hash, dataset_paths, validate_corpus
from run_second_iteration import CORPUS_V2, LATEST, current_state

check_python()


def run_script(script, *arguments):
    stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%S_%fZ")
    log = project_root / "outputs/logs" / (Path(script).stem + "-" + stamp + ".log")
    log.parent.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, "-u", str(project_root / "scripts" / script), *map(str, arguments)]
    print("执行：", script, *arguments)
    print("完整日志：", log)
    process = subprocess.Popen(command, cwd=project_root, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        with log.open("x", encoding="utf-8") as file:
            for line in process.stdout:
                file.write(line)
                file.flush()
                if any(tag in line for tag in ("[RUN]", "[PASS]", "{'loss':", "{'eval_loss':")):
                    print(line.strip(), flush=True)
        code = process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()
    if code:
        print(log.read_text(encoding="utf-8")[-12000:])
        raise RuntimeError(f"{script} 失败；完整日志：{log}，交给 agent 修复")
    return log


Python：3.12.13
解释器：/home/muyi086/github/test_LLM/text-lora/.venv/bin/python3


## 步骤 1：监督新增数据与查看预检查

只展开新增的 26 条，不要求重新审阅第一轮全部样本。动作检查项要求放在旁白；直接话语检查项要求正确说话者和正确次数。它们是人工声明的事实清单，程序不能自动证明所有语义都正确。

数据与两个固定评估集均做来源分离、重复和文本相似度检查。CPU 检查不加载模型权重。沿用第一轮的 rank、学习率、3 轮、512 长度等设置；新增数据意味着优化步数也会增加。

In [2]:
run_script("prepare_training_data.py", "--corpus", CORPUS_V2, "--check-tokens")
samples, _ = validate_corpus(CORPUS_V2)
checks = json.loads(dataset_paths(CORPUS_V2)["checks"].read_text())
print("训练/验证：", sum(s["split"] == "train" for s in samples), sum(s["split"] == "val" for s in samples))
print("完整模板最大 tokens：", max(x["total_tokens"] for x in checks["lengths"]))
blocks = []
for sample in samples:
    if not sample["id"].startswith("r2_"):
        continue
    title = f"{sample['split']} / {sample['id']} / {sample['category']}"
    blocks.append("<details><summary>" + html.escape(title) + "</summary><p>原文：</p><pre>"
                  + html.escape(sample["source_text"]) + "</pre><p>答案：</p><pre>"
                  + html.escape(sample["messages"][-1]["content"]) + "</pre><p>核对："
                  + html.escape("；".join(sample["review_criteria"])) + "</p></details>")
display(HTML("".join(blocks)))
run_script("train_lora.py", "--corpus", CORPUS_V2, "--mode", "full", "--dry-run")
run_script("run_second_iteration.py", "--dry-run")


执行： prepare_training_data.py --corpus /home/muyi086/github/test_LLM/text-lora/datasets/teaching_samples_v2.json --check-tokens
完整日志： /home/muyi086/github/test_LLM/text-lora/outputs/logs/prepare_training_data-20261009T022340_937777Z.log
[PASS] 50 条训练 + 12 条验证；pilot 保留；来源、格式和测试隔离通过。
训练/验证： 50 12
完整模板最大 tokens： 235


执行： train_lora.py --corpus /home/muyi086/github/test_LLM/text-lora/datasets/teaching_samples_v2.json --mode full --dry-run
完整日志： /home/muyi086/github/test_LLM/text-lora/outputs/logs/train_lora-20261009T022345_511002Z.log
[PASS] dry run；未实例化 SftArguments、未加载权重、未创建运行目录。
执行： run_second_iteration.py --dry-run
完整日志： /home/muyi086/github/test_LLM/text-lora/outputs/logs/run_second_iteration-20261009T022349_639032Z.log
[PASS] 计划：CPU模板检查 → 3步试跑 → 3轮全新LoRA → 原10条第二轮推理 → 新10条原模型/第一轮/第二轮推理 → 两份三组对照。未启动GPU任务。


PosixPath('/home/muyi086/github/test_LLM/text-lora/outputs/logs/run_second_iteration-20261009T022349_639032Z.log')

## 步骤 2：采用本轮方案后，运行一次完整实验

下面这个 Cell 会真实使用 GPU，依次自动完成：

1. CPU 模板检查、3 步试跑、从原模型重新初始化的 3 轮 LoRA 训练。
2. 原有 10 条测试的第二轮输出；复用原模型和第一轮的原有输出。
3. 新 10 条未见测试分别运行原模型、第一轮 LoRA、第二轮 LoRA。
4. 核对输入、模型、依赖和解码设置，生成原有测试与新测试两份三组对照。

按独立验证 loss 选择 checkpoint；参考答案和固定测试不用于选 checkpoint。预计需数分钟，以实际日志为准。先停止你自己的其他 GPU 模型服务。失败时保留具体阶段、日志和已完成结果，由 agent 处理。

再次运行会复用完整且数据/配置匹配的第二轮结果。第一轮数据、Notebook 03/04 与 checkpoint 保留。

In [3]:
# 执行此 Cell 即启动第二轮完整实验，无需手动逐条运行脚本。
run_script("run_second_iteration.py")


执行： run_second_iteration.py
完整日志： /home/muyi086/github/test_LLM/text-lora/outputs/logs/run_second_iteration-20261009T022429_053313Z.log
[RUN] checks；完整日志：/home/muyi086/github/test_LLM/text-lora/outputs/iteration2-20261009T022429_919198Z/checks.log
[PASS] 50 条训练 + 12 条验证；pilot 保留；来源、格式和测试隔离通过。
[RUN] train-smoke；完整日志：/home/muyi086/github/test_LLM/text-lora/outputs/iteration2-20261009T022429_919198Z/train-smoke.log
[PASS] 496 个可训练张量，均为文本 LoRA。
{'loss': '0.4306', 'grad_norm': '4.563', 'learning_rate': '0.0001', 'token_acc': '0.8762', 'epoch': '0.16', 'global_step/max_steps': '1/3', 'elapsed_time': '11s', 'remaining_time': '21s', 'memory(GiB)': '9.42', 'train_speed(s/it)': '10.54'}
{'loss': '0.7497', 'grad_norm': '5.047', 'learning_rate': '5e-05', 'token_acc': '0.8126', 'epoch': '0.32', 'global_step/max_steps': '2/3', 'elapsed_time': '22s', 'remaining_time': '11s', 'memory(GiB)': '9.75', 'train_speed(s/it)': '11.19'}
{'loss': '0.129', 'grad_norm': '1.553', 'learning_rate': '0', 'token_acc':

PosixPath('/home/muyi086/github/test_LLM/text-lora/outputs/logs/run_second_iteration-20261009T022429_053313Z.log')

## 步骤 3：查看内容是否改善，并作下一轮决策

优先查看 **新 10 条未见测试**，再看原有测试。双标签比例仅衡量格式；动作完整、直接台词归属、重复次数与普通任务表现都要查看实际输出。参考稿只是保守写法之一，不能把与参考逐字不同直接算成语义错误。

两轮验证集合不同，验证 loss 的绝对值不能直接当作改进比例。两轮都训练 3 轮，但第二轮数据更多、优化步数更多；这是整体迭代对照，不是只隔离某一个变量的因果实验。

In [7]:
if not LATEST.exists():
    print("当前状态：第二轮训练待执行；数据和预检查已完成。采用方案后执行步骤 2，或交给 agent 执行。")
else:
    receipt = json.loads(LATEST.read_text())
    root = Path(receipt["run_dir"])
    metadata = json.loads((root / "iteration_metadata.json").read_text())
    if metadata["status"] != "complete" or metadata["state"] != current_state(CORPUS_V2):
        raise RuntimeError("已有第二轮结果与当前数据/配置不匹配，请交给 agent 核对")
    training = json.loads((root / "train-full/run_metadata.json").read_text())
    print("第二轮目录：", root)
    print("优化步数：", training["result"]["global_step"])
    print("PyTorch 峰值 allocated/reserved GiB：", training["peak_allocated_gib"], training["peak_reserved_gib"])
    print("选中的 adapter：", metadata["second_adapter"])
    display(Markdown("### 新 10 条未见测试"))
    display(Markdown((Path(metadata["fresh_comparison_dir"]) / "comparison.md").read_text()))
    display(Markdown("### 原有 10 条测试"))
    display(Markdown((Path(metadata["fixed_comparison_dir"]) / "comparison.md").read_text()))


第二轮目录： /home/muyi086/github/test_LLM/text-lora/outputs/iteration2-20261009T022429_919198Z
优化步数： 21
PyTorch 峰值 allocated/reserved GiB： 9.142 9.998
选中的 adapter： /home/muyi086/github/test_LLM/text-lora/outputs/iteration2-20261009T022429_919198Z/train-full/checkpoint-21


### 新 10 条未见测试

# 原模型 / 第一轮 / 第二轮输出对照

三组模型配置、输入哈希、依赖版本和解码设置相同，按 ID 对齐。
格式检查只去除开头空 think 块；原始输出仍全部保留。
表中通过只代表双标签与非空正文（允许标签后空格）；内容须阅读原文和检查项。

| ID | 原模型格式 | 第一轮格式 | 第二轮格式 |
| --- | --- | --- | --- |
| r2_test_01 | 通过 | 通过 | 通过 |
| r2_test_02 | 通过 | 通过 | 通过 |
| r2_test_03 | 通过 | 通过 | 通过 |
| r2_test_04 | 通过 | 通过 | 通过 |
| r2_test_05 | 通过 | 通过 | 通过 |
| r2_test_06 | 未通过 | 通过 | 通过 |
| r2_test_07 | 通过 | 通过 | 通过 |
| r2_test_08 | 通过 | 通过 | 通过 |
| r2_test_09 | 见普通任务输出 | 见普通任务输出 | 见普通任务输出 |
| r2_test_10 | 见普通任务输出 | 见普通任务输出 | 见普通任务输出 |

## r2_test_01

原文/任务：

改写下面的文本：
雕塑工坊里，马姝用木楔撑住石座，又把右手移到雕像的肘部，低声说：先取下那条绳。助手解开了绳扣，马姝仍扶着雕像。石座旁落着一小片碎屑。

原模型原始输出：

```text
<think>

</think>

[马姝][沉稳] 先取下那条绳。
[助手][轻柔] （解开绳扣）
[马姝][专注] （仍扶着雕像）
[旁白][客观] 石座旁落着一小片碎屑。
```

第一轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]雕塑工坊里，马姝用木楔撑住石座，又把右手移到雕像的肘部。
[马姝][低声]先取下那条绳。
[助手][平静]（解开了绳扣）
[旁白][平静]马姝仍扶着雕像。石座旁落着一小片碎屑。
```

第二轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]雕塑工坊里，马姝用木楔撑住石座，又把右手移到雕像的肘部。
[马姝][低声]先取下那条绳。
[旁白][平静]助手解开了绳扣，马姝仍扶着雕像。石座旁落着一小片碎屑。
```

训练前冻结的参考写法（其他表达可能同样合理）：

```text
[旁白][平静]雕塑工坊里，马姝用木楔撑住石座，又把右手移到雕像的肘部。
[马姝][低声]先取下那条绳。
[旁白][平静]助手解开了绳扣，马姝仍扶着雕像。
[旁白][平静]石座旁落着一小片碎屑。
```

检查项：

- 说话前的撑石座与移右手都保留
- 低声台词归马姝，仍扶着与碎屑保留

## r2_test_02

原文/任务：

改写下面的文本：
冰球场边，庞舒蹲下系紧鞋带，又将护腿往上推了两厘米，喊：等我一下！队友把球杆靠在围栏上，没有进入冰场。庞舒站起来，鞋带末端仍贴着鞋面。

原模型原始输出：

```text
<think>

</think>

[庞舒][动作] 蹲下系紧鞋带，又将护腿往上推了两厘米
[庞舒][急切] 等我一下！
[队友][无奈] 把球杆靠在围栏上，没有进入冰场
[庞舒][动作] 站起来，鞋带末端仍贴着鞋面
```

第一轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]冰球场边，庞舒蹲下系紧鞋带，又将护腿往上推了两厘米。
[庞舒][喊]等我一下！
[旁白][平静]队友把球杆靠在围栏上，没有进入冰场。
[旁白][平静]庞舒站起来，鞋带末端仍贴着鞋面。
```

第二轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]冰球场边，庞舒蹲下系紧鞋带，又将护腿往上推了两厘米。
[庞舒][喊]等我一下！
[旁白][平静]队友把球杆靠在围栏上，没有进入冰场。
[旁白][平静]庞舒站起来，鞋带末端仍贴着鞋面。
```

训练前冻结的参考写法（其他表达可能同样合理）：

```text
[旁白][平静]冰球场边，庞舒蹲下系紧鞋带，又将护腿往上推了两厘米。
[庞舒][大喊]等我一下！
[旁白][平静]队友把球杆靠在围栏上，没有进入冰场。
[旁白][平静]庞舒站起来，鞋带末端仍贴着鞋面。
```

检查项：

- 保留喊话之前的动作和两厘米
- 没有进入冰场与站起来保留

## r2_test_03

原文/任务：

改写下面的文本：
账房里，杜颐问：这一栏记录的是谁的工时？会计说：是夜班组的。班长说：我还没把加班那张交上来。杜颐在空栏边放了一枚纸夹，会计将表格转向班长。

原模型原始输出：

```text
<think>

</think>

[杜颐][疑惑]这一栏记录的是谁的工时？
[会计][平静]是夜班组的。
[班长][焦急]我还没把加班那张交上来。
[杜颐][果断]（在空栏边放了一枚纸夹）
[会计][动作]（将表格转向班长）
```

第一轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]账房里。
[杜颐][疑问]这一栏记录的是谁的工时？
[会计][平静]是夜班组的。
[班长][平静]我还没把加班那张交上来。
[旁白][平静]杜颐在空栏边放了一枚纸夹，会计将表格转向班长。
```

第二轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]账房里。
[杜颐][疑问]这一栏记录的是谁的工时？
[会计][平静]是夜班组的。
[班长][平静]我还没把加班那张交上来。
[旁白][平静]杜颐在空栏边放了一枚纸夹，会计将表格转向班长。
```

训练前冻结的参考写法（其他表达可能同样合理）：

```text
[旁白][平静]账房里。
[杜颐][疑问]这一栏记录的是谁的工时？
[会计][平静]是夜班组的。
[班长][平静]我还没把加班那张交上来。
[旁白][平静]杜颐在空栏边放了一枚纸夹，会计将表格转向班长。
```

检查项：

- 杜颐的直接问话必须归杜颐
- 夜班组、尚未交加班单和纸夹动作保留

## r2_test_04

原文/任务：

改写下面的文本：
探测站内，沙珩说：屏上有两个亮点。操作员问：右边那个也在移动吗？站长抬起手臂：先记录左边的。沙珩按下记录键，右边的亮点停在刻度七旁。

原模型原始输出：

```text
<think>

</think>

[沙珩][冷静]探测站内，屏上有两个亮点。
[操作员][疑惑]右边那个也在移动吗？
[站长][果断]先记录左边的。
[沙珩][果断]（按下记录键）
[沙珩][平静]右边的亮点停在刻度七旁。
```

第一轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]探测站内。
[沙珩][平静]屏上有两个亮点。
[操作员][疑问]右边那个也在移动吗？
[站长][平静]先记录左边的。
[沙珩][平静]（按下记录键）
[旁白][平静]右边的亮点停在刻度七旁。
```

第二轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]探测站内。
[沙珩][平静]屏上有两个亮点。
[操作员][疑问]右边那个也在移动吗？
[旁白][平静]站长抬起手臂。
[站长][平静]先记录左边的。
[旁白][平静]沙珩按下记录键，右边的亮点停在刻度七旁。
```

训练前冻结的参考写法（其他表达可能同样合理）：

```text
[旁白][平静]探测站内。
[沙珩][平静]屏上有两个亮点。
[操作员][疑问]右边那个也在移动吗？
[旁白][平静]站长抬起手臂。
[站长][平静]先记录左边的。
[旁白][平静]沙珩按下记录键，右边的亮点停在刻度七旁。
```

检查项：

- 站长抬手动作与冒号后台词分离
- 左右、两个和刻度七保留，不替操作员确认运动

## r2_test_05

原文/任务：

改写下面的文本：
织机旁，师傅捏住断开的线头，说：不要换色。柳棠问：备用筒是哪一只？师傅没有指筒子，只重复道：不要换色。柳棠把手伸向两只相同颜色的线筒，又停了下来。

原模型原始输出：

```text
<think>

</think>

[师傅][严肃] 织机旁，师傅捏住断开的线头，说：不要换色。
[柳棠][疑惑] 备用筒是哪一只？
[师傅][重复] 不要换色。
[柳棠][犹豫] 把手伸向两只相同颜色的线筒，又停了下来。
```

第一轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]织机旁，师傅捏住断开的线头。
[师傅][平静]不要换色。
[柳棠][疑问]备用筒是哪一只？
[师傅][平静]不要换色。
[旁白][平静]柳棠把手伸向两只相同颜色的线筒，又停了下来。
```

第二轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]织机旁，师傅捏住断开的线头。
[师傅][平静]不要换色。
[柳棠][疑问]备用筒是哪一只？
[旁白][平静]师傅没有指筒子。
[师傅][平静]不要换色。
[旁白][平静]柳棠把手伸向两只相同颜色的线筒，又停了下来。
```

训练前冻结的参考写法（其他表达可能同样合理）：

```text
[旁白][平静]织机旁，师傅捏住断开的线头。
[师傅][平静]不要换色。
[柳棠][疑问]备用筒是哪一只？
[旁白][平静]师傅没有指筒子。
[师傅][平静]不要换色。
[旁白][平静]柳棠把手伸向两只相同颜色的线筒，又停了下来。
```

检查项：

- 不要换色仅两次且均归师傅
- 不补备用筒是哪只，保留线头和停手

## r2_test_06

原文/任务：

改写下面的文本：
抄写员在便笺上看见一行字：样品转交给了她。晏晴问：写的是负责测量的那位吗？抄写员说：这里没有姓名。晏晴将便笺压在玻璃片下，没有写上任何名字。

原模型原始输出：

```text
<think>

</think>

[旁白] 抄写员在便笺上看见一行字：样品转交给了她。
[晏晴][疑惑] 写的是负责测量的那位吗？
[抄写员][谨慎] 这里没有姓名。
[晏晴][果断] （将便笺压在玻璃片下，没有写上任何名字）
```

第一轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]抄写员在便笺上看见一行字：样品转交给了她。
[晏晴][疑问]写的是负责测量的那位吗？
[抄写员][平静]这里没有姓名。
[旁白][平静]晏晴将便笺压在玻璃片下，没有写上任何名字。
```

第二轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]抄写员在便笺上看见一行字：样品转交给了她。
[晏晴][疑问]写的是负责测量的那位吗？
[抄写员][平静]这里没有姓名。
[旁白][平静]晏晴将便笺压在玻璃片下，没有写上任何名字。
```

训练前冻结的参考写法（其他表达可能同样合理）：

```text
[旁白][平静]抄写员在便笺上看见一行字：样品转交给了她。
[晏晴][疑问]写的是负责测量的那位吗？
[抄写员][平静]这里没有姓名。
[旁白][平静]晏晴将便笺压在玻璃片下，没有写上任何名字。
```

检查项：

- 纸条文字仍是旁白，人物直接问答是对白
- 她保持未知，保留没有写名字

## r2_test_07

原文/任务：

改写下面的文本：
校钟室里，房竹发现秒针每到顶端就停一下。他暗自估计，这可能只是齿轮间隙太大。他没有把估计告诉同事，只在记录卡上画下停顿的位置。

原模型原始输出：

```text
<think>

</think>

[房竹][疑惑] 校钟室里，房竹发现秒针每到顶端就停一下。
[房竹][冷静] 他暗自估计，这可能只是齿轮间隙太大。
[房竹][低沉] 他没有把估计告诉同事，只在记录卡上画下停顿的位置。
```

第一轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]校钟室里，房竹发现秒针每到顶端就停一下。
[旁白][平静]他暗自估计，这可能只是齿轮间隙太大。
[旁白][平静]他没有把估计告诉同事，只在记录卡上画下停顿的位置。
```

第二轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]校钟室里，房竹发现秒针每到顶端就停一下。
[旁白][平静]他暗自估计，这可能只是齿轮间隙太大。
[旁白][平静]他没有把估计告诉同事，只在记录卡上画下停顿的位置。
```

训练前冻结的参考写法（其他表达可能同样合理）：

```text
[旁白][平静]校钟室里，房竹发现秒针每到顶端就停一下。
[旁白][平静]他暗自估计，这可能只是齿轮间隙太大。
[旁白][平静]他没有把估计告诉同事，只在记录卡上画下停顿的位置。
```

检查项：

- 暗自估计保持旁白与可能性
- 停一下和没有告诉同事保留

## r2_test_08

原文/任务：

改写下面的文本：
山口的晾架上挂着六张兽皮。叶颂先取下最左边的一张，再把中间两张翻了面。他将取下的兽皮卷好放进篓中，其余五张仍挂着。架子下的石块没有挪动。

原模型原始输出：

```text
<think>

</think>

[叶颂][平静] 山口的晾架上挂着六张兽皮。
[叶颂][动作] 先取下最左边的一张。
[叶颂][动作] 再把中间两张翻了面。
[叶颂][动作] 将取下的兽皮卷好放进篓中。
[叶颂][动作] 其余五张仍挂着。
[旁白][叙述] 架子下的石块没有挪动。
```

第一轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]山口的晾架上挂着六张兽皮。
[旁白][平静]叶颂先取下最左边的一张，再把中间两张翻了面。
[旁白][平静]他将取下的兽皮卷好放进篓中，其余五张仍挂着。
[旁白][平静]架子下的石块没有挪动。
```

第二轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]山口的晾架上挂着六张兽皮。
[旁白][平静]叶颂先取下最左边的一张，再把中间两张翻了面。
[旁白][平静]他将取下的兽皮卷好放进篓中，其余五张仍挂着。
[旁白][平静]架子下的石块没有挪动。
```

训练前冻结的参考写法（其他表达可能同样合理）：

```text
[旁白][平静]山口的晾架上挂着六张兽皮。
[旁白][平静]叶颂先取下最左边的一张，再把中间两张翻了面。
[旁白][平静]他将取下的兽皮卷好放进篓中，其余五张仍挂着。
[旁白][平静]架子下的石块没有挪动。
```

检查项：

- 全旁白，不添加台词
- 六张、一张、两张、剩五张及先后动作保留

## r2_test_09

原文/任务：

把这句话改成简洁自然的书面表达，只输出一句话：我们这次整理旧档案花的时间比原来估计的多一些，不过现在已经都归到对应的柜子里了。

原模型原始输出：

```text
<think>

</think>

本次旧档案整理耗时略超预估，现已全部归位。
```

第一轮 LoRA原始输出：

```text
<think>

</think>

本次整理旧档案耗时略超预估，现已全部归位。
```

第二轮 LoRA原始输出：

```text
<think>

</think>

本次整理旧档案耗时略超预估，现已全部归位。
```

训练前冻结的参考写法（其他表达可能同样合理）：

```text
此次旧档案整理耗时略超预期，目前已全部归入对应柜中。
```

检查项：

- 按用户指定格式回答
- 保留信息，不使用演绎标签

## r2_test_10

原文/任务：

从记录中提取起始时间和持续时长，每项一行，格式为“起始时间：…”和“持续时长：…”：夜间设备巡检于22点开始，持续四十五分钟，由两人共同完成。

原模型原始输出：

```text
<think>

</think>

起始时间：22点
持续时长：四十五分钟
```

第一轮 LoRA原始输出：

```text
<think>

</think>

起始时间：22点
持续时长：四十五分钟
```

第二轮 LoRA原始输出：

```text
<think>

</think>

起始时间：22点
持续时长：四十五分钟
```

训练前冻结的参考写法（其他表达可能同样合理）：

```text
起始时间：22点
持续时长：四十五分钟
```

检查项：

- 按用户指定格式回答
- 保留信息，不使用演绎标签


### 原有 10 条测试

# 原模型 / 第一轮 / 第二轮输出对照

三组模型配置、输入哈希、依赖版本和解码设置相同，按 ID 对齐。
格式检查只去除开头空 think 块；原始输出仍全部保留。
表中通过只代表双标签与非空正文（允许标签后空格）；内容须阅读原文和检查项。

| ID | 原模型格式 | 第一轮格式 | 第二轮格式 |
| --- | --- | --- | --- |
| tts_01_knock | 未通过 | 通过 | 通过 |
| tts_02_dialogue | 通过 | 通过 | 通过 |
| tts_03_monologue | 通过 | 通过 | 通过 |
| tts_04_three_roles | 未通过 | 通过 | 通过 |
| tts_05_phone | 未通过 | 通过 | 通过 |
| tts_06_action | 未通过 | 通过 | 通过 |
| tts_07_quiet | 通过 | 通过 | 通过 |
| tts_08_ambiguity | 未通过 | 通过 | 通过 |
| control_01_polish | 见普通任务输出 | 见普通任务输出 | 见普通任务输出 |
| control_02_explain | 见普通任务输出 | 见普通任务输出 | 见普通任务输出 |

## tts_01_knock

原文/任务：

改写下面的文本：
走廊尽头传来三下敲门声。林舟停下脚步，低声问：谁在那里？门后没有回答，只有一阵拖动椅子的声音。

原模型原始输出：

```text
<think>

</think>

[旁白] 走廊尽头传来三下敲门声。
[林舟] 低声：谁在那里？
[旁白] 门后没有回答，只有一阵拖动椅子的声音。
```

第一轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]走廊尽头传来三下敲门声。
[林舟][低声]谁在那里？
[旁白][平静]门后没有回答，只有一阵拖动椅子的声音。
```

第二轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]走廊尽头传来三下敲门声。
[旁白][平静]林舟停下脚步。
[林舟][低声]谁在那里？
[旁白][平静]门后没有回答，只有一阵拖动椅子的声音。
```

检查项：

- 保留三下敲门和拖椅子的声音
- 林舟的问话与旁白分开
- 没有补充门后人物身份

## tts_02_dialogue

原文/任务：

改写下面的文本：
沈岚攥着钥匙说：这不是我家的钥匙。周平看了一眼门锁，回答：可它刚刚打开了你的门。沈岚的手开始发抖。

原模型原始输出：

```text
<think>

</think>

[沈岚][惊恐颤抖] 这不是我家的钥匙。
[周平][冷静笃定] 可它刚刚打开了你的门。
[沈岚][极度恐惧] （手剧烈发抖）
```

第一轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]沈岚攥着钥匙。
[沈岚][平静]这不是我家的钥匙。
[旁白][平静]周平看了一眼门锁。
[周平][平静]可它刚刚打开了你的门。
[旁白][平静]沈岚的手开始发抖。
```

第二轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]沈岚攥着钥匙。
[沈岚][平静]这不是我家的钥匙。
[旁白][平静]周平看了一眼门锁。
[周平][平静]可它刚刚打开了你的门。
[旁白][平静]沈岚的手开始发抖。
```

检查项：

- 正确区分沈岚和周平的台词
- 保留钥匙与门锁的矛盾
- 发抖是动作而不是新增台词

## tts_03_monologue

原文/任务：

改写下面的文本：
许川盯着照片。照片里的窗户是开着的，可他清楚记得自己昨晚关过窗。他心想，也许是记错了。他没有把这个想法说出口。

原模型原始输出：

```text
<think>

</think>

[许川][凝重] 许川盯着照片。
[许川][笃定] 照片里的窗户是开着的。
[许川][回忆] 可他清楚记得自己昨晚关过窗。
[许川][疑惑] 他心想，也许是记错了。
[许川][克制] 他没有把这个想法说出口。
```

第一轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]许川盯着照片。
[旁白][平静]照片里的窗户是开着的。
[旁白][平静]可他清楚记得自己昨晚关过窗。
[旁白][平静]他心想，也许是记错了。
[旁白][平静]他没有把这个想法说出口。
```

第二轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]许川盯着照片。
[旁白][平静]照片里的窗户是开着的。
[旁白][平静]他清楚记得自己昨晚关过窗。
[旁白][平静]他心想，也许是记错了。
[旁白][平静]他没有把这个想法说出口。
```

检查项：

- 保留开窗与记忆之间的疑点
- 内心活动不能写成对其他人说出的台词
- 不解释窗户是谁打开的

## tts_04_three_roles

原文/任务：

改写下面的文本：
陈默问：最后一个离开的人是谁？阿宁说：我走的时候，老板还在。老板抬起头：你走后，陈默又回来过。三个人都沉默了。

原模型原始输出：

```text
<think>

</think>

[陈默][疑惑] 最后一个离开的人是谁？
[阿宁][平静] 我走的时候，老板还在。
[老板][低沉] 你走后，陈默又回来过。
[陈默][沉默]
[阿宁][沉默]
[老板][沉默]
```

第一轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]陈默问：最后一个离开的人是谁？
[阿宁][平静]我走的时候，老板还在。
[老板][平静]你走后，陈默又回来过。
[旁白][平静]三个人都沉默了。
```

第二轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]陈默问。
[陈默][疑问]最后一个离开的人是谁？
[阿宁][平静]我走的时候，老板还在。
[老板][平静]你走后，陈默又回来过。
[旁白][平静]三个人都沉默了。
```

检查项：

- 三个人的台词归属正确
- 保留离开后再次回来的信息
- 结尾沉默没有被改成争吵

## tts_05_phone

原文/任务：

改写下面的文本：
电话里，一个陌生男人说：别开灯。叶青屏住呼吸，问他怎么知道自己在家。对方只重复了一遍：别开灯。电话随即挂断。

原模型原始输出：

```text
<think>

</think>

[旁白] 电话铃声突兀地响起，打破了屋内的寂静。
[陌生男人] [低沉压抑] 别开灯。
[叶青] [屏息凝神] 你……怎么知道我在家的？
[陌生男人] [冷漠重复] 别开灯。
[旁白] 话音未落，电话那头传来忙音，随即彻底挂断。
```

第一轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]电话里，一个陌生男人说：别开灯。
[陌生男人][平静]别开灯。
[旁白][平静]叶青屏住呼吸，问他怎么知道自己在家。
[旁白][平静]对方只重复了一遍：别开灯。
[旁白][平静]电话随即挂断。
```

第二轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]电话里，一个陌生男人说。
[陌生男人][平静]别开灯。
[旁白][平静]叶青屏住呼吸，问他怎么知道自己在家。
[陌生男人][平静]别开灯。
[旁白][平静]对方只重复了一遍。
[旁白][平静]电话随即挂断。
```

检查项：

- 两次别开灯都被保留
- 陌生男人保持无名
- 叶青的问题与男人的回答没有混淆

## tts_06_action

原文/任务：

改写下面的文本：
玻璃突然碎了。赵屿扑倒在桌子后，喊：趴下！苏禾蹲在地上，双手护住头。房间里的灯闪了两次，然后彻底熄灭。

原模型原始输出：

```text
<think>

</think>

[旁白] 玻璃突然碎了。
[赵屿][惊恐] 趴下！
[苏禾][惊恐] （蹲在地上，双手护住头）
[旁白] 房间里的灯闪了两次，然后彻底熄灭。
```

第一轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]玻璃突然碎了。
[赵屿][喊]趴下！
[旁白][平静]苏禾蹲在地上，双手护住头。
[旁白][平静]房间里的灯闪了两次，然后彻底熄灭。
```

第二轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]玻璃突然碎了。
[旁白][平静]赵屿扑倒在桌子后。
[赵屿][喊]趴下！
[旁白][平静]苏禾蹲在地上，双手护住头。
[旁白][平静]房间里的灯闪了两次，然后彻底熄灭。
```

检查项：

- 保留动作先后顺序和闪灯两次
- 赵屿喊话使用紧迫的语气
- 不推断是谁打碎了玻璃

## tts_07_quiet

原文/任务：

改写下面的文本：
凌晨两点，陆遥坐在厨房里等水烧开。桌上的另一只杯子还是温的。她轻轻碰了碰杯沿，没有说话。水壶迟迟没有发出声音。

原模型原始输出：

```text
<think>

</think>

[陆遥][低沉压抑] 凌晨两点，我坐在厨房里，就等那壶水烧开。
[陆遥][轻柔试探] 桌上的那只杯子，还留着余温。
[陆遥][小心翼翼] 我轻轻碰了碰杯沿，没敢出声。
[旁白][冷峻] 水壶迟迟没有发出声音。
```

第一轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]凌晨两点，陆遥坐在厨房里等水烧开。
[旁白][平静]桌上的另一只杯子还是温的。
[旁白][平静]她轻轻碰了碰杯沿，没有说话。
[旁白][平静]水壶迟迟没有发出声音。
```

第二轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]凌晨两点，陆遥坐在厨房里等水烧开。
[旁白][平静]桌上的另一只杯子还是温的。
[旁白][平静]她轻轻碰了碰杯沿，没有说话。
[旁白][平静]水壶迟迟没有发出声音。
```

检查项：

- 保留安静场景，没有新增对白
- 温杯子没有被解释为某个人来过
- 演绎标记不过度渲染动作

## tts_08_ambiguity

原文/任务：

改写下面的文本：
老人指着楼上说：他还没走。顾宁问：你说的是昨天那个客人？老人摇了摇头，却没有解释。顾宁听见楼上传来一下脚步声。

原模型原始输出：

```text
<think>

</think>

[老人][严肃] 他还没走。
[顾宁][疑惑] 你说的是昨天那个客人？
[老人][低沉] （摇头）
[顾宁][警觉] （侧耳倾听）
[旁白] 顾宁听见楼上传来一下脚步声。
```

第一轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]老人指着楼上说：他还没走。
[顾宁][疑问]你说的是昨天那个客人？
[旁白][平静]老人摇了摇头，却没有解释。
[旁白][平静]顾宁听见楼上传来一下脚步声。
```

第二轮 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]老人指着楼上。
[老人][平静]他还没走。
[顾宁][疑问]你说的是昨天那个客人？
[旁白][平静]老人摇了摇头，却没有解释。
[旁白][平静]顾宁听见楼上传来一下脚步声。
```

检查项：

- 保留他这一指代的模糊性
- 老人摇头而不是确认客人身份
- 保留一次脚步声，不增加追逐情节

## control_01_polish

原文/任务：

将这句话润色成自然简洁的书面表达，只输出改写后的句子：这个工具我用下来感觉还行，就是有时候等结果等得有点久。

原模型原始输出：

```text
<think>

</think>

该工具使用体验尚可，但响应速度偶有延迟。
```

第一轮 LoRA原始输出：

```text
<think>

</think>

该工具使用体验尚可，但有时结果等待时间较长。
```

第二轮 LoRA原始输出：

```text
<think>

</think>

该工具使用体验尚可，但有时结果等待时间较长。
```

检查项：

- 保留总体满意与等待较久两层意思
- 不使用剧本角色或语气标签
- 只输出一个改写后的句子

## control_02_explain

原文/任务：

用两句话向初学者解释 LoRA 是什么，以及为什么它比全参数微调更节省训练显存。

原模型原始输出：

```text
<think>

</think>

LoRA（Low-Rank Adaptation）是一种通过冻结预训练模型的大部分参数，仅训练少量低秩分解矩阵来适配新任务的微调技术。它之所以比全参数微调更节省显存，是因为它大幅减少了需要更新和存储的参数量，从而显著降低了训练时的内存占用。
```

第一轮 LoRA原始输出：

```text
<think>

</think>

LoRA 是一种通过冻结预训练模型权重并仅训练少量低秩适配器矩阵来实现模型微调的技术。它比全参数微调更节省显存，因为训练时不需要更新庞大的原始模型参数，只需更新少量可训练参数。
```

第二轮 LoRA原始输出：

```text
<think>

</think>

LoRA 是一种通过训练少量可学习参数来微调大型模型的技术，而不是更新整个模型的所有参数。它比全参数微调更节省显存，因为只需要存储和更新这些少量参数，从而大幅减少了训练所需的显存占用。
```

检查项：

- 准确说明低秩适配与冻结大部分原模型参数
- 不声称 LoRA 完全不需要加载原模型
- 不将解释写成有声剧演绎稿


你主要决定是否沿用写法、是否采用第二轮数据、下一轮重点解决哪些实际错误。样本修订、训练执行和逐条内容评审由 agent 完成。训练后先看内容改善与普通任务表现，再决定是否继续扩充；第一轮 checkpoint 留作对照。

细节见 `datasets/iteration2_review.md`。